In [1]:
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import os
import time

In [ ]:
# Constants
RANDOMSTATE = 1082
MODEL_NAME = "MLP Classifier"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [ ]:
EXP = protocol.real_heldout("uav")
feature_names = EXP.feature_names
X_train_scaled = EXP.X_train
X_test_scaled = EXP.X_test
y_train = EXP.y_train
y_test = EXP.y_test


In [5]:
# Train the MLP model
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25),
    activation='tanh',
    solver='adam',
    random_state=RANDOMSTATE,
    learning_rate='adaptive',
    early_stopping=True,
)

start_time = time.time()
mlp.fit(X_train_scaled, y_train)
train_time = time.time() - start_time

In [6]:
# Predict and evaluate
y_pred = mlp.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"MLP Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

MLP Model Accuracy on Test Set: 99.76%
F1 Score: 0.9976
Confusion Matrix:
[[2500    0]
 [   7  391]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      0.99      2898
weighted avg       1.00      1.00      1.00      2898



In [ ]:
from eval_metrics import evaluate, summary, time_inference
from gps_ids.results import append_validated

_proba = mlp.predict_proba(X_test_scaled) if hasattr(mlp, "predict_proba") else None
_timing = time_inference(mlp.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    family="Regular",
    y_proba=_proba,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=mlp.get_params(),
    timing=_timing,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
